In [ ]:
"""
HEALTH WARNING: POSSIBLY INCORRECT PHYSICS BELOW
"""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.animation import FFMpegWriter
from IPython.display import HTML

In [ ]:
# helper functions

def phase(rel):
    """
    Calculate the SR or NR phase of light at all points. Cases differ by a 
    factor of gamma.
    """
    power = 1 if rel else 2
    return 2 * np.pi / lam0 * (-gamma**power * (beta * X + R))

def wavelength(beta, rel):
    """
    Calculate local wavelength of the light in NR or SR case. Cases differ by a
    factor of gamma.
    """
    power = 1 if rel else 2
    return lam0 / (gamma**power * np.sqrt(
        (beta + X / R)**2 + (Y / (R * gamma**2))**2
    ))

def draw_waves(ax, lam, rel):
    """
    Plot 3D phase surface above filled 2D contour plot of wavelength. As well as
    velocity vector
    """

    # qualitative wavelength comparison
    ax.contourf(
        X, Y, lam, offset=-2.5, levels=levels, cmap="jet", vmin=vmin, vmax=vmax
    )

    # qualitative phase comparison
    ax.plot_surface(X, Y, np.cos(phase(rel)), cmap="plasma")

    # source position
    ax.scatter(0, 0, 1, s=50, color="white")

    # velocity vector.
    ax.quiver(
        0, 0, 1, beta, 0, 0, 
        color="white", length=1, 
        normalize=False, linewidth=2, 
        arrow_length_ratio=0.25
    )

def curve_out(vals):
    """
    Converts evenly-distributed values between 0 and 1 to be denser near the
    edges and sparser near the centre of the range.
    """
    return list(vals**2 / (2*vals**2 - 2*vals + 1))

In [ ]:
# Relativistic (SR) and non-relativistic (NR) Doppler effect diagram

# number of frames to sample over in total
frames = 100

# rest wavelength of light
lam0 = 1

# plot initialisation
plt.style.use("dark_background")
fig = plt.figure(figsize=(12,6), constrained_layout=True)
fig.suptitle("Doppler Effect at Relativistic Speeds", fontsize=35, y=0.95)
ax_sr = fig.add_subplot(121, projection="3d", computed_zorder=False)
ax_nr = fig.add_subplot(122, projection="3d", computed_zorder=False)

# data points
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 1, 100)
X, Y = np.meshgrid(x, y)

# animate smoothly over betas
increasing_vals = curve_out(np.linspace(0, 1, frames // 2))
decreasing_vals = curve_out(np.linspace(1, 0, frames // 2))
vals = increasing_vals + decreasing_vals

# calculate ST and NR wavelengths
lam_sr = []
lam_nr = []
for val in vals:
    beta = 0.9 * val
    gamma = 1 / np.sqrt(1 - beta**2)
    R = np.sqrt(X**2 + (Y / gamma)**2)
    lam_sr.append(wavelength(0.9*beta, True))
    lam_nr.append(wavelength(beta, False))

# wavelength range
vmin = min([lams.min() for lams in lam_sr + lam_nr])
vmax = max([lams.max() for lams in lam_sr + lam_nr])
levels = np.linspace(vmin, vmax, 201)

# add colour bar
sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap="jet_r")
cbar = fig.colorbar(
    sm, ax=[ax_sr, ax_nr],
    orientation="horizontal",
    location="bottom",
    shrink=0.5, pad=-0.1
)
cbar.set_ticks([])
cbar.set_label(
    "Qualitative Wavelength (High to Low)", fontsize=15, alpha=0.6
)

def update(frame):

    global beta, gamma, R

    # clear previous frame
    ax_sr.clear()
    ax_nr.clear()

    # format axes
    for ax in [ax_sr, ax_nr]:
        ax.set_aspect("equal", adjustable="datalim")
        ax.axis("off")
        ax.set_box_aspect((2, 1, 1))
        ax.set_zlim(-3, 3)
        ax.view_init(30, -110)
    ax_sr.set_title(
        "Relativistic Effects", fontsize=15, y=0.8, alpha=0.6
    )
    ax_nr.set_title(
        "Non-Relativistic Effects", fontsize=15, y=0.8, alpha=0.6
    )
    
    # computed terms
    beta = 0.9 * vals[frame]
    gamma = 1 / np.sqrt(1 - beta**2)
    R = np.sqrt(X**2 + (Y / gamma)**2)

    # beta label
    ax_sr.text2D(
        0.07, 0.07, rf"$\beta$ = {beta:.3f}", 
        fontsize=25, transform=ax_sr.transAxes
    )
    
    # plot Doppler effect
    draw_waves(ax_sr, lam_sr[frame], True)
    draw_waves(ax_nr, lam_nr[frame], False)

    return []

anim = FuncAnimation(fig, update, frames=frames, blit=False)

# anim.save(f"relativistic-Doppler.gif", writer=PillowWriter(fps=frames/10))

# anim.save(
#     "relativistic-Doppler.mp4", writer=FFMpegWriter(fps=10, bitrate=3000)
# )

plt.close(fig)
# HTML(anim.to_jshtml())